# Join de los dos Datasets que vamos a utilizar (clima y produccion)

## Carga de datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### carga de Produccion

In [ ]:
import os

nombre_buscado = "df_produccion_clean.csv"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

In [ ]:
import pandas as pd

# Ruta exacta donde se encontró el archivo
ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_produccion_clean.csv'

# Cargar el archivo Excel
df_produccion = pd.read_csv(ruta)

# Visualizar las primeras filas
display(df_produccion.shape)
df_produccion.head()

### Carga de Clima


In [ ]:
import os

nombre_buscado = "df_clima_clean.csv"
ruta_encontrada = None

# Buscar el archivo en Google Drive
for raiz, directorios, archivos in os.walk('/content/drive/MyDrive'):
    if nombre_buscado.lower() in [f.lower() for f in archivos]:
        for archivo in archivos:
            if archivo.lower() == nombre_buscado.lower():
                ruta_encontrada = os.path.join(raiz, archivo)
                break

if ruta_encontrada:
    print(f" ARCHIVO ENCONTRADO EN:\n{ruta_encontrada}")
else:
    print(" No se encontró el archivo con ese nombre exacto. Revisa el paso 3.")

In [ ]:
import pandas as pd

# Ruta exacta donde se encontró el archivo
ruta = '/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_clima_clean.csv'

# Cargar el archivo Excel
df_clima = pd.read_csv(ruta)

# Visualizar las primeras filas
display(df_clima.shape)
df_clima.head()

## Union de datasets

In [ ]:
import pandas as pd

df = df_produccion.merge(df_clima, on=['provincia', 'año'], how='left')

print(df.shape)  # (160491, 17)
print(df.isna().sum()[df.isna().sum() > 0])  # columnas climáticas con NaN (años sin dato de clima)

## Recorte y procesamiento del nuevo dataset dataset

In [ ]:
# Opción A: solo años con clima real (recortado)
df_recortado = df.dropna(subset=['T2M'])

# Opción B: todos los años, imputando climáticas faltantes con la media por provincia
cols_clima = ['T2M', 'T2M_MAX', 'T2M_MIN', 'ALLSKY_SFC_SW_DWN', 'RH2M', 'precip_total_mm', 'meses_con_dato']

df_completo = df.copy()
df_completo['tiene_dato_clima'] = df_completo['T2M'].notna().astype(int)

for col in cols_clima:
    df_completo[col] = df_completo.groupby('provincia')[col].transform(lambda x: x.fillna(x.mean()))

In [ ]:
display(df_recortado.shape)
df_recortado.head()

In [ ]:
display(df_completo.shape)
df_completo.head()

In [ ]:
df_completo.isnull().sum()

In [ ]:
df_recortado.isnull().sum()

### Resultado
`df_recortado`: Elimina todas las filas con NaN en clima. Resultado: 111.417 filas × 17 columnas, cubriendo solo 1981-2024. Datos climáticos 100% reales, sin nada inventado. Se pierden ~30% de las filas originales.

`df_completo` : `df_completo['tiene_dato_clima']`crea una columna bandera (1 = dato climático real, 0 = imputado) antes de tocar nada, para no perder el rastro de qué se inventó.
El `for` recorre las 7 columnas climáticas y, para cada una, agrupa por provincia y rellena los NaN con el promedio histórico de esa misma provincia (no un promedio global, para que sea más realista).

## Guardamos los dos Datasets


In [ ]:
df_recortado.to_csv('/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_recortado.csv', index=False)
df_completo.to_csv('/content/drive/MyDrive/Proyecto Agro (Grupo 1)/Colabs/data/df_completo.csv', index=False)
